# ROOT File Structure Inspector

Inspect keys, TTrees, branches, and branch presence (`patternVec`) for selected ROOT files.

In [1]:
import ROOT as r

PI_ROOT_DICT_PATH = "/simulation/docker/install/lib/libPiRootDict.so"

root_files = {
    "truth_hst": "/simulation/workdir_link/playground/reco_algorithm_tests/data/all_truth_hst.root",
    "truth_rec": "/simulation/workdir_link/playground/reco_algorithm_tests/data/all_truth_rec.root",
    "reco_rec": "/simulation/workdir_link/playground/reco_algorithm_tests/data/all_data_rec.root",
}

print("Loading dictionary:", PI_ROOT_DICT_PATH)
status = r.gSystem.Load(PI_ROOT_DICT_PATH)
print("gSystem.Load status:", status)

Module libc not found.


Loading dictionary: /simulation/docker/install/lib/libPiRootDict.so
gSystem.Load status: 0


In [2]:
def open_root(path):
    f = r.TFile.Open(path)
    if not f or f.IsZombie():
        raise RuntimeError(f"Failed to open ROOT file: {path}")
    return f


def list_keys(f):
    print(f"\n=== Keys in {f.GetName()} ===")
    keys = f.GetListOfKeys()
    if not keys or keys.GetSize() == 0:
        print("(no keys)")
        return
    for k in keys:
        print(f"- {k.GetName()}  [{k.GetClassName()}]")


def collect_ttrees(f):
    trees = []
    keys = f.GetListOfKeys()
    if not keys:
        return trees
    for k in keys:
        name = k.GetName()
        obj = f.Get(name)
        if obj and obj.InheritsFrom("TTree"):
            trees.append(obj)
    return trees


def print_tree_summary(tree, max_branches=120):
    print(f"\nTree: {tree.GetName()}  Entries: {tree.GetEntries()}")
    branches = tree.GetListOfBranches()
    if not branches:
        print("  (no branches)")
        return
    n = branches.GetSize()
    print(f"  Branches ({n} total):")
    for i, b in enumerate(branches):
        if i >= max_branches:
            print(f"  ... ({n - max_branches} more)")
            break
        print(f"  - {b.GetName()}")

In [3]:
opened = {}
file_trees = {}

for label, path in root_files.items():
    f = open_root(path)
    opened[label] = f
    list_keys(f)
    trees = collect_ttrees(f)
    file_trees[label] = trees
    print(f"\nTrees in {label}:", [t.GetName() for t in trees])


=== Keys in /simulation/workdir_link/playground/reco_algorithm_tests/data/all_truth_hst.root ===
- noat_all_box_00  [TH3F]
- noat_all_nopp_00  [TH3F]
- noat_all_dedz_00  [TH3F]
- noat_pienu_box_00  [TH3F]
- noat_pienu_nopp_00  [TH3F]
- noat_pienu_dedz_00  [TH3F]
- noat_pimu_box_00  [TH3F]
- noat_pimu_nopp_00  [TH3F]
- noat_pimu_dedz_00  [TH3F]
- noat_mudif_box_00  [TH3F]
- noat_mudif_nopp_00  [TH3F]
- noat_mudif_dedz_00  [TH3F]
- noat_other_box_00  [TH3F]
- noat_other_nopp_00  [TH3F]
- noat_other_dedz_00  [TH3F]
- acc_all_box_00  [TH3F]
- acc_all_nopp_00  [TH3F]
- acc_all_dedz_00  [TH3F]
- acc_pienu_box_00  [TH3F]
- acc_pienu_nopp_00  [TH3F]
- acc_pienu_dedz_00  [TH3F]
- acc_pimu_box_00  [TH3F]
- acc_pimu_nopp_00  [TH3F]
- acc_pimu_dedz_00  [TH3F]
- acc_mudif_box_00  [TH3F]
- acc_mudif_nopp_00  [TH3F]
- acc_mudif_dedz_00  [TH3F]
- acc_other_box_00  [TH3F]
- acc_other_nopp_00  [TH3F]
- acc_other_dedz_00  [TH3F]
- shift_all_box_00  [TH2F]
- shift_all_nopp_00  [TH2F]
- shift_all_dedz_00 

In [4]:
for label, trees in file_trees.items():
    print(f"\n{'=' * 18} {label} {'=' * 18}")
    for t in trees:
        print_tree_summary(t)


================== truth_hst ==================

================== truth_rec ==================

Tree: rec  Entries: 7388
  Branches (16 total):
  - infoVec
  - dtarHitsVec
  - atarHitsVec
  - trackerHitsVec
  - caloHitsVec
  - tagVec
  - summaryVec
  - patternVec
  - trackletVec
  - clusterVec

================== reco_rec ==================

Tree: rec  Entries: 7388
  Branches (16 total):
  - infoVec
  - dtarHitsVec
  - atarHitsVec
  - trackerHitsVec
  - caloHitsVec
  - tagVec
  - summaryVec
  - patternVec
  - trackletVec
  - clusterVec


In [5]:
def has_branch(tree, name):
    return bool(tree.GetBranch(name))

for label, trees in file_trees.items():
    print(f"\n=== patternVec check: {label} ===")
    for t in trees:
        print(f"{t.GetName():<32} patternVec: {has_branch(t, 'patternVec')}")


=== patternVec check: truth_hst ===

=== patternVec check: truth_rec ===
rec                              patternVec: True

=== patternVec check: reco_rec ===
rec                              patternVec: True


In [6]:
# Optional: deep inspect one tree schema
# Example: selected = ("reco_rec", "PIONEER_DER")
selected = None

if selected:
    label, tree_name = selected
    src = opened[label].Get(tree_name)
    if src and src.InheritsFrom("TTree"):
        print_tree_summary(src, max_branches=300)
        print("\nROOT schema print:")
        src.Print()
    else:
        print(f"Tree not found: {selected}")